# SHL Hiring Assessment 2026 — Grammar Scoring Engine

I score each spoken answer for grammar, from 0 to 5. The clips are about a minute long.

I have 769 training clips with real scores. The file I submit is `sample_submission.csv` (204 rows). `test.csv` has 216 files. Its labels are random, so I do not use them.

The leaderboard uses Pearson correlation and RMSE. This notebook has my approach, the steps I took, the model, the plots, and the training RMSE.


## Approach

The guide is about sentences, not accent. A low score means short or broken sentences. A high score means correct, longer sentences, and the speaker fixes a slip when they hear it.

I do not score the raw audio. With 769 clips a sound model mostly learns the voice. I turn each clip into text and measure the things in the guide.

I make two transcripts.

Whisper large-v3-turbo is good at word times and it adds full stops. It also rewrites messy grammar into clean English, so I do not use it as the grammar text. I use it for pauses, repeats, and sentence shape.

wav2vec2 writes what it hears and does not tidy the grammar. I measure mistakes on that text.

What I count:

- Mistakes: how many words a grammar-correction model changes, and how acceptable each sentence looks (CoLA). I also keep the worst sentence.
- If Whisper and wav2vec2 disagree, I store that gap. I keep it only if the 5-fold check says it helps the score.
- Sentence shape: length, depth of the parse, and clauses per sentence. This uses the Whisper text, because it has full stops.
- Unfinished sentences: no verb, three words or fewer, or the clip ends on a word like "and" or "the".
- Fixing yourself: "I mean", "sorry", false starts, and the same word twice in a row.
- Pauses, and fillers like "um" and "uh".
- Better complex grammar: more clauses and fewer corrections. The tree model can use both at once.

I also build a small MiniLM embedding of the text. I mix it with the main model only when the 5-fold check says the mix is better.

I use 5 folds. Every file is named `audio_<id>.wav`, so I cannot split by speaker. The score 1.0 has only one clip, so a stratified split does not work. I use a shuffled 5-fold. The mix weight is chosen on the other folds, not on the fold I am scoring. Predictions are kept between 0 and 5.

The training RMSE is the error on clips the model has already seen. It will look too good. The 5-fold RMSE and Pearson are what I trust for the leaderboard.


## Preprocessing

I match each row to its wav by file name.

`test.csv` has a label column. Those numbers are placeholders. I drop them.

`sample_submission.csv` fixes the row order. 99 of those names are not in the zip. I have no audio for them, so those rows get the average training score. The other 105 clips are in the data, and I score them from the audio.

1. Whisper, with word times. I turn `condition_on_previous_text` off so one sentence does not pull the next one off course.
2. I read the same clip as mono, resample to 16 kHz, and run wav2vec2. It writes in capitals, so I change it back to normal case before the grammar models see it.
3. Pauses, speed, fillers, repeats, and self-corrections come from the Whisper word times.
4. Sentence shape comes from the Whisper text. A one-minute wav2vec2 transcript usually has no full stops, so it would look like one huge sentence.
5. Acceptability and the grammar-edit rate come from the wav2vec2 text. Where there is no punctuation, I cut the text into short windows.
6. I save the transcripts and the features. The next run does not transcribe again.

## Setup

Open this notebook from the project folder, or from `notebooks/`. The first pass over the 769 training clips takes several hours on my Mac. After that, fitting the model takes a few minutes.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise FileNotFoundError("Open or run this notebook from the project directory.")
sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import pandas as pd
from IPython.display import Image, Markdown, display

from src.data import DataNotFoundError
from src.pipeline import run

print("Project:", ROOT)


## Metrics

RMSE is how big the mistakes are. A large miss counts more than a small one. Pearson checks whether a higher human score gets a higher prediction.

I pick the model with both: Pearson minus RMSE divided by 5.

The task asks for the training RMSE in this notebook. That is the error of the final model on the training clips. It is optimistic. The 5-fold RMSE and Pearson are closer to the leaderboard.


In [ ]:
import numpy as np
from src.metrics import pearson, rmse

# Scores are cut to 0-5 before either metric.
# RMSE is the square root of the average squared error.
# Pearson is the correlation between the real score and my prediction.
demo_y = np.array([1.0, 2.0, 3.0, 4.0])
demo_p = np.array([1.2, 1.8, 3.4, 3.6])
print(f"Demo RMSE {rmse(demo_y, demo_p):.4f}")
print(f"Demo Pearson {pearson(demo_y, demo_p):.4f}")
print("A perfect prediction has RMSE 0 and Pearson 1.")
print(f"Perfect RMSE {rmse(demo_y, demo_y):.4f}   Perfect Pearson {pearson(demo_y, demo_y):.4f}")


## Train

The next cell does the full run: match the wavs, transcribe, build features, check with 5 folds, fit on all 769 clips, and write `outputs/submission.csv`.

The line that starts with `TRAINING RMSE` is the number this task asks for.


In [ ]:
try:
    metrics = run()
except DataNotFoundError as exc:
    metrics = None
    print(exc)
    print()
    print("Put the competition files in data/ and run this cell again.")
    print("A zip in data/ is unpacked on its own.")
    print("Or run: python scripts/download_competition.py")


## Results

For the 5-fold numbers, each clip is scored by a model that did not train on it. The training numbers come from the final model, after it has seen every training clip.


In [ ]:
metrics_path = ROOT / "outputs" / "metrics.json"
if not metrics_path.exists():
    print("No numbers yet. Run the training cell after the audio is in data/.")
else:
    metrics = json.loads(metrics_path.read_text())
    print(metrics["banner"])
    print()
    print(f"Main model: {metrics['tabular_model']}")
    print(f"Weight on the hand-built features: {metrics['blend_weight_tabular']:.2f}")
    print(f"Extra calibration used: {metrics['calibration']['used']}")
    print(f"Folds: {metrics['fold_kind']}    features: {metrics['n_features']}")
    print(f"Train clips: {metrics['n_train']}    rows to submit: {metrics['n_test']}")
    print()
    rows = []
    for name, scores in metrics["model_scores"].items():
        rows.append({"model": name, "CV RMSE": scores["rmse"], "CV Pearson": scores["pearson"]})
    comparison = pd.DataFrame(rows).sort_values("CV RMSE")
    display(comparison.round(4))


## Figures

I plot the training scores, how each measurement moves with the score, predicted score against real score, and which measurements the model uses.

If a measurement is doing the right job, more grammar edits and more unfinished sentences should sit with lower scores, and more clauses should sit with higher scores. The training scatter will look tighter than the 5-fold scatter, because the model has already seen those clips.


In [ ]:
figure_dir = ROOT / "outputs" / "figures"
if not figure_dir.exists() or not any(figure_dir.glob("*.png")):
    print("Plots show up after training.")
else:
    order = [
        "label_distribution.png",
        "feature_correlations.png",
        "gec_wer_vs_score.png",
        "cola_vs_score.png",
        "fragment_rate_vs_score.png",
        "clauses_vs_score.png",
        "pred_vs_actual_cv.png",
        "pred_vs_actual_train.png",
        "residuals.png",
        "feature_importance.png",
    ]
    captions = {
        "label_distribution.png": "How the training scores are spread. The model has to cover the whole range.",
        "feature_correlations.png": "How strongly each measurement lines up with the grammar score.",
        "gec_wer_vs_score.png": "How much the grammar fixer had to change the transcript.",
        "cola_vs_score.png": "Average chance that a sentence is grammatically ok.",
        "fragment_rate_vs_score.png": "Share of sentences with no verb. This is the unfinished-sentence clue.",
        "clauses_vs_score.png": "Dependent clauses per sentence. This is the complex-grammar clue.",
        "pred_vs_actual_cv.png": "5-fold predictions. This is the plot I use to guess the leaderboard.",
        "pred_vs_actual_train.png": "Predictions on the training clips. The training RMSE comes from these points.",
        "residuals.png": "Real score minus prediction. A bend here would mean the model is off on part of the 0 to 5 range.",
        "feature_importance.png": "How much the error grows if I shuffle one feature on a held-out fold.",
    }
    for name in order:
        path = figure_dir / name
        if not path.exists():
            continue
        display(Markdown(f"**{captions.get(name, name)}**"))
        display(Image(filename=str(path)))


## Low scores and high scores

A few transcripts from the low end and the high end of the training scores. The prediction next to the label is from the 5-fold check, so that clip was not in the training fold. I read these to check that I am scoring grammar, not the topic or how long the clip is.


In [ ]:
examples_path = ROOT / "outputs" / "examples.json"
if not examples_path.exists():
    print("Examples show up after training.")
else:
    examples = json.loads(examples_path.read_text())
    for example in examples:
        print("=" * 72)
        print(
            f"{example['band'].upper()}   id={example['audio_id']}   "
            f"label={example['label']:.2f}   oof prediction={example['oof_prediction']:.2f}"
        )
        print("Grammar transcript:")
        print(example.get("grammar_transcript") or "(empty)")
        print()


## Evaluation results

**TRAINING RMSE (required): 0.5349**

**TRAINING Pearson r: 0.9041**

**5-fold CV RMSE: 0.6392**

**5-fold CV Pearson r: 0.8573**

Main model: `hgb_shallow`. Weight on the hand-built features: 1.00. Extra calibration: no. Folds: shuffled 5-fold. Features: 57. Train clips: 769. Rows to submit: 204.

Training RMSE is the error on clips the model has already seen. The 5-fold RMSE and Pearson are closer to the leaderboard.

99 rows have no wav in the zip. Those scores are the average training score.

## Requirement checklist

| What was asked | Where it is |
| --- | --- |
| Score from 0 to 5 | Predictions are cut to that range before `outputs/submission.csv` is written. |
| Preprocessing explained | The section above, and `src/transcribe.py`. |
| Pipeline explained | The approach section above. |
| RMSE and Pearson | `src/metrics.py`, the metrics section, and the results section. |
| Training RMSE in this notebook | The Evaluation results section, as **TRAINING RMSE (required)**. The training cell prints it too. |
| A cross-check, not only the training error | 5-fold RMSE and Pearson in the same printout. The average-score baseline is in the comparison table. |
| Plots | Score histogram, feature correlations, a few rubric scatters, predicted vs actual for the 5-fold and the training fit, residuals, feature importance. |
| Do not use `test.csv` labels | `src/data.py` does not read them as targets. |
| Submission file | `outputs/submission.csv`, same columns and same row order as `sample_submission.csv`. |

The training cell has to finish on the real audio before the training RMSE is a number.


## Submission

`outputs/submission.csv` follows `sample_submission.csv`: same id column, same row order. Scores stay between 0 and 5.

## How I would explain this

The score is a grammar score, so I start from the words.

Whisper writes tidy English, so it hides the mistakes I am meant to score. wav2vec2 does not. I measure how much a grammar model still has to change, and whether each sentence looks acceptable.

Pauses, false starts, and unfinished sentences cover the rest of the guide. I add a small text embedding only if the 5-fold check says it helps both the ranking and the size of the error.

I report the training RMSE because the task asks for it. The 5-fold RMSE and Pearson are the numbers I would use to guess the leaderboard.
